# Notebook 04 — Hyperparameter Optimisation
**Mental Health Assessment Using Machine Learning**

---

## Purpose

This notebook applies three hyperparameter search strategies — Randomized Search, Grid Search, and Bayesian Optimisation — to all 8 algorithms across all 9 feature sets, producing 216 optimised models (72 per strategy). All results are saved to disk.

---

## Search Strategies

| Strategy | Class | Iterations | Scoring | Notes |
|----------|-------|-----------|---------|-------|
| Randomized Search | `RandomizedSearchCV` | 30 random draws | macro F1 | Fast broad exploration over full parameter ranges |
| Grid Search | `GridSearchCV` | Exhaustive over focused grid | macro F1 | Narrower, deterministic — informed by random search ranges |
| Bayesian Optimisation | `BayesSearchCV` | 30 iterations | macro F1 | Sequential guided search; most sample-efficient |

All three strategies use `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` to ensure class proportions are preserved across folds. Macro F1 is used as the optimisation target (not accuracy) to avoid bias toward the majority class (Critical: 64% of test set).

**n_jobs handling:** `RandomizedSearchCV` and `GridSearchCV` use `n_jobs=-1` to parallelise CV folds. For `BayesSearchCV`, `n_jobs=1` is used to avoid nested parallelism issues; the sklearn estimators inside each iteration already use their own thread pools.

---

## Hyperparameter Search Spaces

### Randomized Search — full exploration ranges

| Algorithm | Parameters |
|-----------|----------|
| LR | `C` ∈ {0.01,0.1,1,10,100} · `solver` ∈ {lbfgs,saga} · `max_iter` ∈ {500,1000,2000} |
| DT | `max_depth` ∈ [3,20] · `min_samples_split` ∈ [2,20] · `min_samples_leaf` ∈ [1,10] · `criterion` ∈ {gini,entropy} |
| RF | `n_estimators` ∈ {50,100,200,300,500} · `max_depth` ∈ {5,10,15,20,30,None} · `min_samples_split` ∈ {2,5,10} · `max_features` ∈ {sqrt,log2} |
| KNN | `n_neighbors` ∈ [3,20] · `weights` ∈ {uniform,distance} · `metric` ∈ {euclidean,manhattan} |
| SVM | `C` ∈ {0.1,1,10,100} · `kernel` ∈ {rbf,linear,poly} · `gamma` ∈ {scale,auto,0.001,0.01,0.1} |
| GB | `n_estimators` ∈ {50,100,200,300} · `max_depth` ∈ [3,8] · `learning_rate` ∈ {0.01,0.05,0.1,0.2,0.3} · `min_samples_split` ∈ {2,5,10} |
| XGB | `n_estimators` ∈ {50,100,200,300} · `max_depth` ∈ [3,8] · `learning_rate` ∈ {0.01,0.05,0.1,0.2,0.3} · `subsample` ∈ {0.6,0.7,0.8,0.9,1.0} · `colsample_bytree` ∈ {0.6,0.7,0.8,0.9,1.0} |
| LGBM | `n_estimators` ∈ {50,100,200,300} · `num_leaves` ∈ {20,31,50,70,100} · `learning_rate` ∈ {0.01,0.05,0.1,0.2,0.3} · `min_child_samples` ∈ {10,20,30,50} |

### Grid Search — focused grid

| Algorithm | Parameters |
|-----------|----------|
| LR | `C` ∈ {0.1,1,10} · `solver` ∈ {lbfgs,saga} |
| DT | `max_depth` ∈ {5,10,15} · `min_samples_split` ∈ {2,5,10} · `criterion` ∈ {gini,entropy} |
| RF | `n_estimators` ∈ {100,200,300} · `max_depth` ∈ {10,20,None} · `max_features` ∈ {sqrt,log2} |
| KNN | `n_neighbors` ∈ {3,5,7,11} · `weights` ∈ {uniform,distance} |
| SVM | `C` ∈ {1,10,100} · `kernel` ∈ {rbf,linear} · `gamma` ∈ {scale,auto} |
| GB | `n_estimators` ∈ {100,200} · `max_depth` ∈ {3,5} · `learning_rate` ∈ {0.05,0.1,0.2} |
| XGB | `n_estimators` ∈ {100,200} · `max_depth` ∈ {3,5,6} · `learning_rate` ∈ {0.05,0.1,0.2} |
| LGBM | `n_estimators` ∈ {100,200} · `num_leaves` ∈ {31,50} · `learning_rate` ∈ {0.05,0.1,0.2} |

### Bayesian Optimisation — continuous/integer spaces

| Algorithm | Parameters |
|-----------|----------|
| LR | `C` ∈ Real(0.01,100,log-uniform) · `max_iter` ∈ Integer(500,2000) |
| DT | `max_depth` ∈ Integer(3,20) · `min_samples_split` ∈ Integer(2,20) · `min_samples_leaf` ∈ Integer(1,10) · `criterion` ∈ Categorical |
| RF | `n_estimators` ∈ Integer(50,500) · `max_depth` ∈ Integer(5,30) · `min_samples_split` ∈ Integer(2,10) · `max_features` ∈ Categorical |
| KNN | `n_neighbors` ∈ Integer(3,20) · `weights` ∈ Categorical · `metric` ∈ Categorical |
| SVM | `C` ∈ Real(0.1,100,log-uniform) · `gamma` ∈ Real(0.001,0.1,log-uniform) · `kernel` ∈ Categorical |
| GB | `n_estimators` ∈ Integer(50,300) · `max_depth` ∈ Integer(3,8) · `learning_rate` ∈ Real(0.01,0.3,log-uniform) |
| XGB | `n_estimators` ∈ Integer(50,300) · `max_depth` ∈ Integer(3,8) · `learning_rate` ∈ Real(0.01,0.3,log-uniform) · `subsample` ∈ Real(0.6,1.0) · `colsample_bytree` ∈ Real(0.6,1.0) |
| LGBM | `n_estimators` ∈ Integer(50,300) · `num_leaves` ∈ Integer(20,100) · `learning_rate` ∈ Real(0.01,0.3,log-uniform) · `min_child_samples` ∈ Integer(10,50) |

---

## Cell Map

| Cell | Summary |
|------|---------|
| 1 | Imports · load all 9 feature sets · define CV and scoring · define base algorithms |
| 2 | Define all three search spaces (Randomized, Grid, Bayesian) |
| 3 | RandomizedSearchCV × 8 algorithms × 9 feature sets · save models + per-method CSVs |
| 4 | Save Randomized summary CSV · print top 10 |
| 5 | GridSearchCV × 8 × 9 · save models + per-method CSVs |
| 6 | Save Grid summary CSV · print top 10 |
| 7 | BayesSearchCV × 8 × 9 · save models + per-method CSVs |
| 8 | Save Bayes summary CSV · print top 10 · notebook complete |

## Cell 1 — Imports, Load Feature Sets, Define CV and Base Algorithms

Loads all 9 scaled feature sets from disk (produced by Notebook 02). Defines the `StratifiedKFold` cross-validator and the base algorithm instances used as templates for all three search strategies. To avoid nested parallelism issues on Windows, base algorithms are instantiated with `n_jobs=1`; parallelism is applied at the search level instead.

In [1]:
from pathlib import Path
import os, warnings, copy
warnings.filterwarnings('ignore')

_cwd = Path.cwd()
if _cwd.name == 'notebooks':
    os.chdir(_cwd.parent)
print(f'Working directory: {Path.cwd()}')

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import (
    RandomizedSearchCV, GridSearchCV, StratifiedKFold, cross_val_score
)
from sklearn.ensemble       import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model   import LogisticRegression
from sklearn.tree           import DecisionTreeClassifier
from sklearn.neighbors      import KNeighborsClassifier
from sklearn.svm            import SVC
from sklearn.metrics        import accuracy_score, f1_score
from skopt                  import BayesSearchCV
from skopt.space            import Real, Integer, Categorical
import xgboost  as xgb
import lightgbm as lgb

METHODS     = ['rfe', 'skb', 'fscs', 'etc', 'pc', 'mi', 'mir', 'mu', 'vt']
SEED        = 42
CV          = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
SCORING     = 'f1_macro'

# ── Load all 9 feature sets from disk ─────────────────────────────────────────
feature_sets = {}
for method in METHODS:
    train_df = pd.read_csv(os.path.join('features', 'Tabular', method, 'train.csv'))
    test_df  = pd.read_csv(os.path.join('features', 'Tabular', method, 'test.csv'))
    feat_cols = [c for c in train_df.columns if c != 'label']
    feature_sets[method] = {
        'train_X': train_df[feat_cols].values,
        'train_y': train_df['label'].values.astype(int),
        'test_X' : test_df[feat_cols].values,
        'test_y' : test_df['label'].values.astype(int),
    }

# Base algorithms — n_jobs=1 to avoid nested parallelism in SearchCV
BASE_ALGORITHMS = {
    'lr'  : LogisticRegression(max_iter=1000, random_state=SEED, n_jobs=1),
    'dt'  : DecisionTreeClassifier(random_state=SEED),
    'rf'  : RandomForestClassifier(random_state=SEED, n_jobs=1),
    'knn' : KNeighborsClassifier(n_jobs=1),
    'svm' : SVC(probability=True, random_state=SEED),
    'gb'  : GradientBoostingClassifier(random_state=SEED),
    'xgb' : xgb.XGBClassifier(random_state=SEED, n_jobs=1,
                               eval_metric='mlogloss', verbosity=0),
    'lgbm': lgb.LGBMClassifier(random_state=SEED, n_jobs=1, verbose=-1),
}

print(f'✓ {len(feature_sets)} feature sets loaded')
for m, fd in feature_sets.items():
    print(f'  {m.upper():<5}: train={fd["train_X"].shape}  test={fd["test_X"].shape}')
print(f'\nCV       : StratifiedKFold(n_splits=5)')
print(f'Scoring  : {SCORING}')
print('\n✓ Setup complete')

Working directory: d:\Programming\Projects\Mental Health Assessment
✓ 9 feature sets loaded
  RFE  : train=(3099, 15)  test=(405, 15)
  SKB  : train=(3099, 15)  test=(405, 15)
  FSCS : train=(3099, 15)  test=(405, 15)
  ETC  : train=(3099, 15)  test=(405, 15)
  PC   : train=(3099, 15)  test=(405, 15)
  MI   : train=(3099, 15)  test=(405, 15)
  MIR  : train=(3099, 15)  test=(405, 15)
  MU   : train=(3099, 15)  test=(405, 15)
  VT   : train=(3099, 15)  test=(405, 15)

CV       : StratifiedKFold(n_splits=5)
Scoring  : f1_macro

✓ Setup complete


## Cell 2 — Define All Three Search Spaces

Defines parameter distributions for Randomized Search, parameter grids for Grid Search, and continuous/integer/categorical spaces for Bayesian Optimisation. All three dicts are keyed by the same algorithm short names as `BASE_ALGORITHMS`. Grid Search uses a focused, narrower grid informed by the ranges explored in Randomized Search — this keeps the combinatorial explosion manageable while still covering the most promising region.

In [2]:
# ── Randomized Search — full exploration ranges ────────────────────────────────
RANDOM_PARAMS = {
    'lr'  : {
        'C'       : [0.01, 0.1, 1, 10, 100],
        'solver'  : ['lbfgs', 'saga'],
        'max_iter': [500, 1000, 2000],
    },
    'dt'  : {
        'max_depth'        : list(range(3, 21)),
        'min_samples_split': list(range(2, 21)),
        'min_samples_leaf' : list(range(1, 11)),
        'criterion'        : ['gini', 'entropy'],
    },
    'rf'  : {
        'n_estimators'     : [50, 100, 200, 300, 500],
        'max_depth'        : [5, 10, 15, 20, 30, None],
        'min_samples_split': [2, 5, 10],
        'max_features'     : ['sqrt', 'log2'],
    },
    'knn' : {
        'n_neighbors': list(range(3, 21)),
        'weights'    : ['uniform', 'distance'],
        'metric'     : ['euclidean', 'manhattan'],
    },
    'svm' : {
        'C'     : [0.1, 1, 10, 100],
        'kernel': ['rbf', 'linear', 'poly'],
        'gamma' : ['scale', 'auto', 0.001, 0.01, 0.1],
    },
    'gb'  : {
        'n_estimators'     : [50, 100, 200, 300],
        'max_depth'        : list(range(3, 9)),
        'learning_rate'    : [0.01, 0.05, 0.1, 0.2, 0.3],
        'min_samples_split': [2, 5, 10],
    },
    'xgb' : {
        'n_estimators'   : [50, 100, 200, 300],
        'max_depth'      : list(range(3, 9)),
        'learning_rate'  : [0.01, 0.05, 0.1, 0.2, 0.3],
        'subsample'      : [0.6, 0.7, 0.8, 0.9, 1.0],
        'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    },
    'lgbm': {
        'n_estimators'    : [50, 100, 200, 300],
        'num_leaves'      : [20, 31, 50, 70, 100],
        'learning_rate'   : [0.01, 0.05, 0.1, 0.2, 0.3],
        'min_child_samples': [10, 20, 30, 50],
    },
}

# ── Grid Search — focused grid ─────────────────────────────────────────────────
GRID_PARAMS = {
    'lr'  : {'C': [0.1, 1, 10], 'solver': ['lbfgs', 'saga']},
    'dt'  : {
        'max_depth'        : [5, 10, 15],
        'min_samples_split': [2, 5, 10],
        'criterion'        : ['gini', 'entropy'],
    },
    'rf'  : {
        'n_estimators': [100, 200, 300],
        'max_depth'   : [10, 20, None],
        'max_features': ['sqrt', 'log2'],
    },
    'knn' : {'n_neighbors': [3, 5, 7, 11], 'weights': ['uniform', 'distance']},
    'svm' : {
        'C'     : [1, 10, 100],
        'kernel': ['rbf', 'linear'],
        'gamma' : ['scale', 'auto'],
    },
    'gb'  : {
        'n_estimators' : [100, 200],
        'max_depth'    : [3, 5],
        'learning_rate': [0.05, 0.1, 0.2],
    },
    'xgb' : {
        'n_estimators' : [100, 200],
        'max_depth'    : [3, 5, 6],
        'learning_rate': [0.05, 0.1, 0.2],
    },
    'lgbm': {
        'n_estimators' : [100, 200],
        'num_leaves'   : [31, 50],
        'learning_rate': [0.05, 0.1, 0.2],
    },
}

# ── Bayesian Optimisation — continuous/integer/categorical spaces ──────────────
BAYES_PARAMS = {
    'lr'  : {
        'C'       : Real(0.01, 100, prior='log-uniform'),
        'max_iter': Integer(500, 2000),
    },
    'dt'  : {
        'max_depth'        : Integer(3, 20),
        'min_samples_split': Integer(2, 20),
        'min_samples_leaf' : Integer(1, 10),
        'criterion'        : Categorical(['gini', 'entropy']),
    },
    'rf'  : {
        'n_estimators'     : Integer(50, 500),
        'max_depth'        : Integer(5, 30),
        'min_samples_split': Integer(2, 10),
        'max_features'     : Categorical(['sqrt', 'log2']),
    },
    'knn' : {
        'n_neighbors': Integer(3, 20),
        'weights'    : Categorical(['uniform', 'distance']),
        'metric'     : Categorical(['euclidean', 'manhattan']),
    },
    'svm' : {
        'C'     : Real(0.1, 100, prior='log-uniform'),
        'gamma' : Real(0.001, 0.1, prior='log-uniform'),
        'kernel': Categorical(['rbf', 'linear']),
    },
    'gb'  : {
        'n_estimators' : Integer(50, 300),
        'max_depth'    : Integer(3, 8),
        'learning_rate': Real(0.01, 0.3, prior='log-uniform'),
    },
    'xgb' : {
        'n_estimators'    : Integer(50, 300),
        'max_depth'       : Integer(3, 8),
        'learning_rate'   : Real(0.01, 0.3, prior='log-uniform'),
        'subsample'       : Real(0.6, 1.0),
        'colsample_bytree': Real(0.6, 1.0),
    },
    'lgbm': {
        'n_estimators'    : Integer(50, 300),
        'num_leaves'      : Integer(20, 100),
        'learning_rate'   : Real(0.01, 0.3, prior='log-uniform'),
        'min_child_samples': Integer(10, 50),
    },
}

print('✓ Search spaces defined')
print(f'  Randomized : {len(RANDOM_PARAMS)} algorithms')
print(f'  Grid       : {len(GRID_PARAMS)} algorithms')
print(f'  Bayesian   : {len(BAYES_PARAMS)} algorithms')

✓ Search spaces defined
  Randomized : 8 algorithms
  Grid       : 8 algorithms
  Bayesian   : 8 algorithms


## Cell 3 — RandomizedSearchCV × 8 Algorithms × 9 Feature Sets (72 Models)

Runs `RandomizedSearchCV` with 30 random draws for each algorithm–feature set combination. For each combination, the best estimator (refitted on the full training set) is evaluated on the held-out test set. The best hyperparameters, CV score (mean macro F1 across 5 folds), test accuracy, and test macro F1 are recorded. The best estimator is saved as `{algo}_randomized.pkl`.

In [3]:
def run_search(search_obj, algo_name, method, X_tr, y_tr, X_te, y_te, suffix, model_dir):
    """Fit a search object, evaluate on test set, save model, return result dict."""
    search_obj.fit(X_tr, y_tr)
    best_clf   = search_obj.best_estimator_
    y_pred     = best_clf.predict(X_te)
    cv_score   = search_obj.best_score_
    test_acc   = accuracy_score(y_te, y_pred)
    test_f1m   = f1_score(y_te, y_pred, average='macro', zero_division=0)
    print(f'  {algo_name.upper():<5}: CV_F1={cv_score:.4f}  Test_Acc={test_acc:.4f}  Test_F1_macro={test_f1m:.4f}')
    os.makedirs(model_dir, exist_ok=True)
    joblib.dump(best_clf, os.path.join(model_dir, f'{algo_name}_{suffix}.pkl'))
    return {
        'Feature_Method': method,
        'Model'         : algo_name.upper(),
        'Best_Params'   : str(search_obj.best_params_),
        'CV_F1_Macro'   : round(cv_score, 4),
        'Test_Accuracy' : round(test_acc,  4),
        'Test_F1_Macro' : round(test_f1m,  4),
    }


all_random_results = []

for method, fd in feature_sets.items():
    print(f'\n── RANDOMIZED | {method.upper()} ──')
    X_tr, y_tr = fd['train_X'], fd['train_y']
    X_te, y_te = fd['test_X'],  fd['test_y']
    method_results = []
    model_dir = os.path.join('models', 'Optimised', method)

    for algo_name, base_algo in BASE_ALGORITHMS.items():
        search = RandomizedSearchCV(
            estimator          = copy.deepcopy(base_algo),
            param_distributions= RANDOM_PARAMS[algo_name],
            n_iter             = 30,
            scoring            = SCORING,
            cv                 = CV,
            n_jobs             = -1,
            random_state       = SEED,
            refit              = True,
        )
        row = run_search(search, algo_name, method, X_tr, y_tr, X_te, y_te,
                         'randomized', model_dir)
        method_results.append(row)
        all_random_results.append(row)

    res_dir = os.path.join('results', 'Optimised', method)
    os.makedirs(res_dir, exist_ok=True)
    pd.DataFrame(method_results).to_csv(
        os.path.join(res_dir, 'randomized_search_results.csv'), index=False
    )

print('\n✓ RandomizedSearchCV complete — 72 models saved')


── RANDOMIZED | RFE ──
  LR   : CV_F1=0.8883  Test_Acc=0.8420  Test_F1_macro=0.7690
  DT   : CV_F1=0.9084  Test_Acc=0.8593  Test_F1_macro=0.7971
  RF   : CV_F1=0.9531  Test_Acc=0.9012  Test_F1_macro=0.8454
  KNN  : CV_F1=0.9540  Test_Acc=0.9037  Test_F1_macro=0.8531
  SVM  : CV_F1=0.9560  Test_Acc=0.8914  Test_F1_macro=0.8190
  GB   : CV_F1=0.9521  Test_Acc=0.8914  Test_F1_macro=0.8415
  XGB  : CV_F1=0.9561  Test_Acc=0.8938  Test_F1_macro=0.8455
  LGBM : CV_F1=0.9547  Test_Acc=0.8963  Test_F1_macro=0.8405

── RANDOMIZED | SKB ──
  LR   : CV_F1=0.8856  Test_Acc=0.8444  Test_F1_macro=0.7645
  DT   : CV_F1=0.9076  Test_Acc=0.8667  Test_F1_macro=0.8297
  RF   : CV_F1=0.9528  Test_Acc=0.9086  Test_F1_macro=0.8601
  KNN  : CV_F1=0.9481  Test_Acc=0.8914  Test_F1_macro=0.8367
  SVM  : CV_F1=0.9566  Test_Acc=0.8889  Test_F1_macro=0.8081
  GB   : CV_F1=0.9502  Test_Acc=0.8938  Test_F1_macro=0.8385
  XGB  : CV_F1=0.9521  Test_Acc=0.9012  Test_F1_macro=0.8482
  LGBM : CV_F1=0.9528  Test_Acc=0.898

## Cell 4 — Save Randomized Search Summary CSV

Aggregates all 72 Randomized Search results into a single DataFrame and saves it to `summary/Results/Optimised/randomized_results.csv`. The top 10 results by Test_F1_Macro are printed to surface the most promising algorithm–feature method combinations before the Grid Search runs.

In [4]:
random_summary = pd.DataFrame(all_random_results)
os.makedirs(os.path.join('summary', 'Results', 'Optimised'), exist_ok=True)
RAND_SUMMARY = os.path.join('summary', 'Results', 'Optimised', 'randomized_results.csv')
random_summary.to_csv(RAND_SUMMARY, index=False)

print(f'✓ Saved : {RAND_SUMMARY}')
print(f'  Shape : {random_summary.shape}  (expected 72 rows)')
print()
print('Top 10 by Test_F1_Macro:')
print(
    random_summary
    .sort_values('Test_F1_Macro', ascending=False)
    [['Feature_Method', 'Model', 'CV_F1_Macro', 'Test_Accuracy', 'Test_F1_Macro']]
    .head(10)
    .to_string(index=False)
)

✓ Saved : summary\Results\Optimised\randomized_results.csv
  Shape : (72, 6)  (expected 72 rows)

Top 10 by Test_F1_Macro:
Feature_Method Model  CV_F1_Macro  Test_Accuracy  Test_F1_Macro
            mi    GB       0.9496         0.9111         0.8601
           mir    GB       0.9496         0.9111         0.8601
            pc    RF       0.9528         0.9086         0.8601
           skb    RF       0.9528         0.9086         0.8601
            mu   XGB       0.9561         0.9037         0.8582
           rfe   KNN       0.9540         0.9037         0.8531
          fscs   XGB       0.9506         0.9037         0.8522
            mu  LGBM       0.9541         0.9111         0.8513
          fscs    GB       0.9535         0.8963         0.8508
            mu    RF       0.9532         0.9062         0.8506


## Cell 5 — GridSearchCV × 8 Algorithms × 9 Feature Sets (72 Models)

Runs exhaustive `GridSearchCV` over the focused parameter grids defined in Cell 2. The grid is narrow enough to be computationally feasible while covering the most informative hyperparameter combinations. The same `run_search` helper is reused. Each best estimator is saved as `{algo}_grid.pkl`.

In [5]:
all_grid_results = []

for method, fd in feature_sets.items():
    print(f'\n── GRID | {method.upper()} ──')
    X_tr, y_tr = fd['train_X'], fd['train_y']
    X_te, y_te = fd['test_X'],  fd['test_y']
    method_results = []
    model_dir = os.path.join('models', 'Optimised', method)

    for algo_name, base_algo in BASE_ALGORITHMS.items():
        search = GridSearchCV(
            estimator = copy.deepcopy(base_algo),
            param_grid= GRID_PARAMS[algo_name],
            scoring   = SCORING,
            cv        = CV,
            n_jobs    = -1,
            refit     = True,
        )
        row = run_search(search, algo_name, method, X_tr, y_tr, X_te, y_te,
                         'grid', model_dir)
        method_results.append(row)
        all_grid_results.append(row)

    res_dir = os.path.join('results', 'Optimised', method)
    os.makedirs(res_dir, exist_ok=True)
    pd.DataFrame(method_results).to_csv(
        os.path.join(res_dir, 'grid_search_results.csv'), index=False
    )

print('\n✓ GridSearchCV complete — 72 models saved')


── GRID | RFE ──
  LR   : CV_F1=0.8866  Test_Acc=0.8444  Test_F1_macro=0.7671
  DT   : CV_F1=0.9085  Test_Acc=0.8667  Test_F1_macro=0.8009
  RF   : CV_F1=0.9531  Test_Acc=0.9012  Test_F1_macro=0.8454
  KNN  : CV_F1=0.9469  Test_Acc=0.8741  Test_F1_macro=0.8161
  SVM  : CV_F1=0.9557  Test_Acc=0.9062  Test_F1_macro=0.8481
  GB   : CV_F1=0.9534  Test_Acc=0.8938  Test_F1_macro=0.8482
  XGB  : CV_F1=0.9563  Test_Acc=0.8815  Test_F1_macro=0.8205
  LGBM : CV_F1=0.9551  Test_Acc=0.8963  Test_F1_macro=0.8435

── GRID | SKB ──
  LR   : CV_F1=0.8856  Test_Acc=0.8444  Test_F1_macro=0.7645
  DT   : CV_F1=0.9097  Test_Acc=0.8519  Test_F1_macro=0.7947
  RF   : CV_F1=0.9525  Test_Acc=0.9086  Test_F1_macro=0.8520
  KNN  : CV_F1=0.9459  Test_Acc=0.8864  Test_F1_macro=0.8230
  SVM  : CV_F1=0.9566  Test_Acc=0.8889  Test_F1_macro=0.8081
  GB   : CV_F1=0.9486  Test_Acc=0.9037  Test_F1_macro=0.8428
  XGB  : CV_F1=0.9537  Test_Acc=0.9086  Test_F1_macro=0.8471
  LGBM : CV_F1=0.9522  Test_Acc=0.9086  Test_F1_m

## Cell 6 — Save Grid Search Summary CSV

Aggregates all 72 Grid Search results and saves to `summary/Results/Optimised/grid_results.csv`. Top 10 by Test_F1_Macro are printed.

In [6]:
grid_summary = pd.DataFrame(all_grid_results)
GRID_SUMMARY = os.path.join('summary', 'Results', 'Optimised', 'grid_results.csv')
grid_summary.to_csv(GRID_SUMMARY, index=False)

print(f'✓ Saved : {GRID_SUMMARY}')
print(f'  Shape : {grid_summary.shape}  (expected 72 rows)')
print()
print('Top 10 by Test_F1_Macro:')
print(
    grid_summary
    .sort_values('Test_F1_Macro', ascending=False)
    [['Feature_Method', 'Model', 'CV_F1_Macro', 'Test_Accuracy', 'Test_F1_Macro']]
    .head(10)
    .to_string(index=False)
)

✓ Saved : summary\Results\Optimised\grid_results.csv
  Shape : (72, 6)  (expected 72 rows)

Top 10 by Test_F1_Macro:
Feature_Method Model  CV_F1_Macro  Test_Accuracy  Test_F1_Macro
            mu   XGB       0.9522         0.9037         0.8608
            mi   XGB       0.9522         0.9012         0.8584
           mir   XGB       0.9522         0.9012         0.8584
            mu    GB       0.9538         0.9086         0.8549
          fscs   XGB       0.9496         0.8988         0.8526
          fscs    GB       0.9506         0.8988         0.8523
            pc    RF       0.9525         0.9086         0.8520
           skb    RF       0.9525         0.9086         0.8520
            pc  LGBM       0.9522         0.9086         0.8502
           skb  LGBM       0.9522         0.9086         0.8502


## Cell 7 — BayesSearchCV × 8 Algorithms × 9 Feature Sets (72 Models)

Runs `BayesSearchCV` (scikit-optimize) with 30 iterations per combination. Bayesian optimisation builds a surrogate model of the objective function and uses it to select the next hyperparameter configuration to evaluate — making it more sample-efficient than random search. `n_jobs=1` is set for the search object to avoid nested parallelism issues; the sklearn estimators' internal thread pools handle their own parallelism. Each best estimator is saved as `{algo}_bayes.pkl`.

In [7]:
all_bayes_results = []

for method, fd in feature_sets.items():
    print(f'\n── BAYES | {method.upper()} ──')
    X_tr, y_tr = fd['train_X'], fd['train_y']
    X_te, y_te = fd['test_X'],  fd['test_y']
    method_results = []
    model_dir = os.path.join('models', 'Optimised', method)

    for algo_name, base_algo in BASE_ALGORITHMS.items():
        search = BayesSearchCV(
            estimator   = copy.deepcopy(base_algo),
            search_spaces= BAYES_PARAMS[algo_name],
            n_iter      = 30,
            scoring     = SCORING,
            cv          = CV,
            n_jobs      = 1,
            random_state= SEED,
            refit       = True,
        )
        row = run_search(search, algo_name, method, X_tr, y_tr, X_te, y_te,
                         'bayes', model_dir)
        method_results.append(row)
        all_bayes_results.append(row)

    res_dir = os.path.join('results', 'Optimised', method)
    os.makedirs(res_dir, exist_ok=True)
    pd.DataFrame(method_results).to_csv(
        os.path.join(res_dir, 'bayes_search_results.csv'), index=False
    )

print('\n✓ BayesSearchCV complete — 72 models saved')


── BAYES | RFE ──
  LR   : CV_F1=0.8893  Test_Acc=0.8420  Test_F1_macro=0.7690
  DT   : CV_F1=0.9110  Test_Acc=0.8519  Test_F1_macro=0.7909
  RF   : CV_F1=0.9535  Test_Acc=0.9037  Test_F1_macro=0.8475
  KNN  : CV_F1=0.9540  Test_Acc=0.9037  Test_F1_macro=0.8531
  SVM  : CV_F1=0.9605  Test_Acc=0.9062  Test_F1_macro=0.8283
  GB   : CV_F1=0.9512  Test_Acc=0.8840  Test_F1_macro=0.8339
  XGB  : CV_F1=0.9564  Test_Acc=0.8938  Test_F1_macro=0.8389
  LGBM : CV_F1=0.9548  Test_Acc=0.8914  Test_F1_macro=0.8325

── BAYES | SKB ──
  LR   : CV_F1=0.8858  Test_Acc=0.8420  Test_F1_macro=0.7627
  DT   : CV_F1=0.9102  Test_Acc=0.8568  Test_F1_macro=0.8039
  RF   : CV_F1=0.9522  Test_Acc=0.9062  Test_F1_macro=0.8501
  KNN  : CV_F1=0.9484  Test_Acc=0.8815  Test_F1_macro=0.8258
  SVM  : CV_F1=0.9628  Test_Acc=0.8765  Test_F1_macro=0.7814
  GB   : CV_F1=0.9502  Test_Acc=0.8988  Test_F1_macro=0.8319
  XGB  : CV_F1=0.9537  Test_Acc=0.9086  Test_F1_macro=0.8667
  LGBM : CV_F1=0.9528  Test_Acc=0.9086  Test_F1

## Cell 8 — Save Bayes Summary CSV and Notebook Complete

Aggregates all 72 Bayes Search results and saves to `summary/Results/Optimised/bayes_results.csv`. The top 10 results by Test_F1_Macro are printed. A completion checklist summarises all files produced by this notebook.

In [8]:
bayes_summary = pd.DataFrame(all_bayes_results)
BAYES_SUMMARY = os.path.join('summary', 'Results', 'Optimised', 'bayes_results.csv')
bayes_summary.to_csv(BAYES_SUMMARY, index=False)

print(f'✓ Saved : {BAYES_SUMMARY}')
print(f'  Shape : {bayes_summary.shape}  (expected 72 rows)')
print()
print('Top 10 by Test_F1_Macro:')
print(
    bayes_summary
    .sort_values('Test_F1_Macro', ascending=False)
    [['Feature_Method', 'Model', 'CV_F1_Macro', 'Test_Accuracy', 'Test_F1_Macro']]
    .head(10)
    .to_string(index=False)
)
print()
print('── Notebook 04 complete ──')
print('  models/Optimised/          216 .pkl models  (72 randomized + 72 grid + 72 bayes)')
print('  results/Optimised/          27 per-method CSVs  (3 strategies × 9 methods)')
print('  summary/Results/Optimised/  randomized_results.csv')
print('                              grid_results.csv')
print('                              bayes_results.csv')

✓ Saved : summary\Results\Optimised\bayes_results.csv
  Shape : (72, 6)  (expected 72 rows)

Top 10 by Test_F1_Macro:
Feature_Method Model  CV_F1_Macro  Test_Accuracy  Test_F1_Macro
            pc   XGB       0.9537         0.9086         0.8667
           skb   XGB       0.9537         0.9086         0.8667
          fscs  LGBM       0.9562         0.9111         0.8665
            mi   XGB       0.9529         0.9086         0.8591
           mir   XGB       0.9529         0.9086         0.8591
          fscs    RF       0.9532         0.9012         0.8586
           rfe   KNN       0.9540         0.9037         0.8531
          fscs   XGB       0.9538         0.8988         0.8523
            mi    RF       0.9529         0.9037         0.8510
           mir    RF       0.9529         0.9037         0.8510

── Notebook 04 complete ──
  models/Optimised/          216 .pkl models  (72 randomized + 72 grid + 72 bayes)
  results/Optimised/          27 per-method CSVs  (3 strategies × 9